# Gradient Descent vs. Scikit-Learn

This notebook tests a custom gradient-descent algorithm imported from `gradient_descent.py`. The model is trained on the `auto_mpg` dataset to predict fuel consumption (MPG) from vehicle weight and engine displacement.

We compare against scikit-learn's `LinearRegression` to show that the underlying mathematics matches a production solver in accuracy.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from gradient_descent import LineaireRegressie

# Cleaner plot styling
sns.set_theme(style="whitegrid")

## Step 1: Load and split the data

We use the `auto_mpg` dataset and split it into a training set (80% of the cars, used for fitting) and a test set (20% of the cars, used to evaluate unseen data).


In [ ]:
# Load the data
auto_df = sns.load_dataset('mpg').dropna()

# Select features (X) and target (y)
X = auto_df[['weight', 'displacement']]
y = np.array(auto_df['mpg'])

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)}")
print(f"Test set size:     {len(X_test)}")


## Step 2: Train the models
Both models are fit on exactly the same training data. For the custom gradient-descent model we set a very small learning rate (step size). Vehicle weight values around 3,500 lbs produce steep gradients; a small step size prevents the optimizer from diverging.


In [ ]:
# 1. Custom model
mijn_model = LineaireRegressie()

mijn_model.fit(
    X=X_train, 
    y=y_train, 
    a10=-0.005, 
    a20=-0.03, 
    b0=38.0, 
    lr=0.00001,  
    num_steps=10000
)

score_eigen_model = mijn_model.score(X_test, y_test)

In [ ]:
# 2. Scikit-learn model
sklearn_model = LinearRegression()
sklearn_model.fit(X_train, y_train)

y_pred_sklearn = sklearn_model.predict(X_test)
score_sklearn = root_mean_squared_error(y_test, y_pred_sklearn)

In [ ]:
# Results
print("Comparison:")
print(f"Custom model RMSE:      {score_eigen_model:.4f} MPG")
print(f"Scikit-learn RMSE:      {score_sklearn:.4f} MPG")
print("-" * 46)
print("Learned parameters:")
print(f"Custom model:   a1={mijn_model.a1:.6f}, a2={mijn_model.a2:.6f}, b={mijn_model.b:.2f}")
print(f"Scikit-learn:   a1={sklearn_model.coef_[0]:.6f}, a2={sklearn_model.coef_[1]:.6f}, b={sklearn_model.intercept_:.2f}")


Both models perform well and are nearly indistinguishable. The custom gradient-descent model has an RMSE of approximately 4.14 MPG, meaning predictions deviate from the true fuel efficiency by about 4.14 miles per gallon on average. On this particular test split it even slightly outperforms scikit-learn (4.28 MPG).


## Step 3: Visual comparison
To check whether the predictions are equivalent, we plot them against the true test-set values. The closer the points lie to the diagonal red line, the better the prediction.


In [ ]:
# Generate predictions for the plots
voorspellingen_eigen = mijn_model.predict(X_test)

# Create the figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: custom model
sns.scatterplot(x=y_test, y=voorspellingen_eigen, ax=ax1, color='blue', alpha=0.7)
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
ax1.set_title('Custom Gradient Descent Model', fontsize=14, fontweight='bold')
ax1.set_xlabel('True MPG')
ax1.set_ylabel('Predicted MPG')

# Plot 2: scikit-learn model
sns.scatterplot(x=y_test, y=y_pred_sklearn, ax=ax2, color='green', alpha=0.7)
ax2.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
ax2.set_title('Scikit-Learn LinearRegression', fontsize=14, fontweight='bold')
ax2.set_xlabel('True MPG')
ax2.set_ylabel('Predicted MPG')

plt.tight_layout()
plt.show()


A notable pattern in both scatter plots is that the models systematically underpredict the most efficient cars (true MPG above 30). Predicted values fall consistently below the ideal red line, meaning those vehicles are treated as less efficient than they actually are. This highlights a fundamental limitation of linear regression: the algorithm fits a single hyperplane through the cloud of points, whereas the true relationship between weight, displacement, and fuel consumption is likely non-linear in the high-efficiency tail.
